In [1]:
import numpy as np
import pandas as pd
import ast
import re
from sentence_transformers import SentenceTransformer, util

/Users/basmala.ayman/Documents/FCIS/GP/GitHub Repo/Hackathons-Team-Formation-pre-processing/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Members Embeddings

In [2]:
members = pd.read_csv('datasets/mapped_members.csv')

In [3]:
members.head(3)

,username,bio,hard_skills,hackathons_interests,projects_count,hackathons_count,achievements_count,followers_count,following_count,likes_count,winnings_count,Descriptions,hard_skills_clean,mapped_soft_skills
0,dembiczakmatt,NaN,[],[],2,2,9,0,0,1,2,[],[],[]
1,pradeepkprakasam,Blockchain Developer,"['javascript', 'solidity', 'react', 'react-nat...","['Blockchain', 'Education', 'Fintech', 'IoT', ...",1,6,6,4,2,0,1,"[""I learned a lot about Self Sovereign Identit...","['react', 'python', 'javascript', 'solidity', ...",[]
2,janet-han,Software Engineer,"['html', 'css', 'javascript', 'ajax', 'apis', ...","['DevOps', 'Productivity']",1,3,6,1,0,0,1,"['Software Engineer, Blockchain Developer']","['mongodb', 'ajax', 'css', 'apis', 'ruby-on-ra...",[]


In [4]:
members.shape

(121134, 14)

In [5]:
members['hard_skills'].apply(type).value_counts()

hard_skills
<class 'str'>    121134
Name: count, dtype: int64

## Convert String Columns to Lists

In [6]:
def to_list(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    x = str(x).strip()
    if x == "" or x == "[]":
        return []
    try:
        return ast.literal_eval(x)
    except:
        return []

In [7]:
members['hard_skills'] = members['hard_skills'].apply(to_list)
members['Descriptions'] = members['Descriptions'].apply(to_list)
members['hard_skills_clean'] = members['hard_skills_clean'].apply(to_list)
members['mapped_soft_skills'] = members['mapped_soft_skills'].apply(to_list)

In [8]:
members['mapped_soft_skills'].apply(type).value_counts()

mapped_soft_skills
<class 'list'>    121134
Name: count, dtype: int64

In [9]:
members['bio'] = members['bio'].fillna('')

## Clean Text

In [10]:
def clean_text(text):
  text = str(text).lower()
  text = re.sub(r"http\S+", " ", text)
  text = re.sub(r"[_|,/]", " ", text)
  text = re.sub(r"\s+", " ", text).strip()
  return text

In [11]:
def clean_skill(token):
    token = str(token).lower().strip()
    token = re.sub(r"\s+", " ", token)
    token = re.sub(r"[^a-z0-9\+\#\.\- ]", " ", token) # keep + # . (for c++, c#, node.js)
    token = re.sub(r"\s+", " ", token).strip()
    return token

In [12]:
def uniq_clean_list(lst):
    if not isinstance(lst, list):
        return []
    out = []
    seen = set()
    for x in lst:
        x = clean_skill(x)
        if x and x not in seen:
            out.append(x)
            seen.add(x)
    return out

### Clean skills from duplications

In [13]:
members['hard_skills_clean'] = members['hard_skills_clean'].apply(uniq_clean_list)
members['mapped_soft_skills'] = members['mapped_soft_skills'].apply(uniq_clean_list)

## Build only one text field for each member

In [14]:
def build_member_text(member_row):
  bio = clean_text(member_row.get("bio", ""))
  hard = member_row.get('hard_skills_clean', [])
  soft = member_row.get('mapped_soft_skills', [])
  desc_list = [clean_text(d) for d in member_row.get("Descriptions", [])]

  # convert lists into text
  hard_text = ', '.join(hard) if hard else ""
  soft_text = ', '.join(soft) if soft else ""
  desc_text = ' '.join(desc_list) if desc_list else ""

  parts = []
  if bio:
    parts.append(f"bio: {bio}")
  if hard_text:
    parts.append(f"hard skills: {hard_text}")
    parts.append(f"hard skills: {hard_text}") # to give them larger wight
  if soft_text:
    parts.append(f"soft skills: {soft_text}")
  if desc_text:
    parts.append(f"experience: {desc_text}")

  if not parts:
    return "no profile"

  return " | ".join(parts)

In [15]:
members['member_text'] = members.apply(build_member_text, axis=1)

In [16]:
members[['bio', 'hard_skills_clean', 'mapped_soft_skills', 'member_text']].head()

,bio,hard_skills_clean,mapped_soft_skills,member_text
0,,[],[],no profile
1,Blockchain Developer,"[react, python, javascript, solidity, react-na...",[],bio: blockchain developer | hard skills: react...
2,Software Engineer,"[mongodb, ajax, css, apis, ruby-on-rails, pyth...",[],"bio: software engineer | hard skills: mongodb,..."
3,,"[unity, html5, css, python, javascript, java, c]",[],"hard skills: unity, html5, css, python, javasc..."
4,"Media, blockchain, IP lawyer and founder of A...","[modelling, business]",[],bio: media blockchain ip lawyer and founder of...


## Generate Embeddings

In [18]:
# for macOs
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="mps")
## for windows
# model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1889.42it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
members = members.reset_index(drop=True)
texts = members['member_text'].tolist()

In [20]:
embeddings = model.encode(
    texts,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True # makes cosine similarity easier later
)

Batches: 100%|██████████| 947/947 [02:31<00:00,  6.23it/s]


## Save embeddings as npy file

In [22]:
np.save('embeddings/member_embeddings.npy', embeddings)
members["username"].to_csv('embeddings/member_ids.csv', index=False)

In [24]:
emb = np.load('embeddings/member_embeddings.npy')
ids = pd.read_csv('embeddings/member_ids.csv')

print(emb.shape)
print(len(ids))

(121134, 384)
121134


# Projects Embeddings

In [26]:
teams = pd.read_csv('datasets/teams_clean.csv')

In [27]:
teams.shape

(35830, 8)

In [28]:
teams['winners_description'].isna().sum()

np.int64(27432)

In [29]:
teams = teams.drop(columns=['members_count','hackathon_slug'])

## Handel project tags

In [30]:
teams['project_tags'].apply(type).value_counts()

project_tags
<class 'str'>    35830
Name: count, dtype: int64

In [31]:
teams['project_tags']=teams['project_tags'].apply(to_list)

In [32]:
teams['project_tags'].apply(type).value_counts()

project_tags
<class 'list'>    35830
Name: count, dtype: int64

### Clean project tags from duplications

In [33]:
teams['project_tags']=teams['project_tags'].apply(uniq_clean_list)

### Merge all tags as a string

In [34]:
def list_to_text(lst):
  if not lst:
    return ""
  return ", ".join(lst)

In [35]:
teams['project_tags']=teams['project_tags'].apply(list_to_text)

In [36]:
teams['project_tags'].head(4)

0    beautiful-soup, bootstrap, css, flask, html, p...
1    centos, docker, flask, javascript, natural-lan...
2    css3, django, javascript, keras, python, tenso...
3    css, docker, firebase, html, javascript, linod...
Name: project_tags, dtype: str

In [37]:
teams["short_description"] = teams["short_description"].fillna("")
teams["winners_description"] = teams["winners_description"].fillna("")

## Build one text field for each project

In [38]:
def winner_label(x):
    if str(x).lower() in ["true", "1", "yes"]:
        return "winner"
    return "non winner"

In [39]:
teams["winner_text"] = teams["is_winner"].apply(winner_label)

In [40]:
teams[['is_winner', 'winner_text']].head(3)

,is_winner,winner_text
0,True,winner
1,True,winner
2,True,winner


In [41]:
def build_project_text(team_row):
  tags = clean_text(team_row.get("project_tags", ""))
  desc = clean_text(team_row.get("short_description", ""))
  win_desc = clean_text(team_row.get("winners_description", ""))
  winner = team_row.get("winner_text", "non winner")

  parts = []
  if tags:
      parts.append(f"project was built with: {tags}")
  if desc:
      parts.append(f"description: {desc}")
  parts.append(f"project is: {winner}")
  if win_desc:
      parts.append(f"award info: {win_desc}")

  return " | ".join(parts)

In [42]:
teams["project_text"] = teams.apply(build_project_text, axis=1)

In [43]:
teams.head()

,project_slug,is_winner,winners_description,project_tags,project_description,short_description,winner_text,project_text
0,player-value-rating,True,1st Place Overall\n,"beautiful-soup, bootstrap, css, flask, html, p...",Although various of inequities are present in ...,Ontario basketball athletes are often overlook...,winner,project was built with: beautiful-soup bootstr...
1,linkhack,True,Education\n,"centos, docker, flask, javascript, natural-lan...",Inspiration Our inspiration for this project i...,LinkHack is a React web application served by ...,winner,project was built with: centos docker flask ja...
2,tpay,True,Synopsys Sponsor Prize - Synops...,"css3, django, javascript, keras, python, tenso...",Inspiration Forgetting to carry your Tcard and...,"Get rid of your Tcard, pay with your face",winner,project was built with: css3 django javascript...
3,winter-arch,True,Philly Codefest 2022 Student Te...,"css, docker, firebase, html, javascript, linod...",Inspiration Philadelphia is facing the greates...,"In 2020, our city lost 1200 lives to opioid ov...",winner,project was built with: css docker firebase ht...
4,quizeria,True,Best UI/UX Award\n,"ajax, canva, css, html5, javascript, ui, ux, velo","Inspiration When many of us were young, we lov...",Bring out the whiz in you.,winner,project was built with: ajax canva css html5 j...


## Build Embeddings

In [44]:
teams = teams.reset_index(drop=True)
project_texts = teams["project_text"].tolist()

In [45]:
project_embeddings = model.encode(
    project_texts,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches: 100%|██████████| 280/280 [02:42<00:00,  1.73it/s]


## Save Result

In [46]:
np.save(
    "embeddings/project_embeddings.npy",
    project_embeddings
)

teams["project_slug"].to_csv(
    "embeddings/project_ids.csv",
    index=False
)

In [47]:
pro_emb = np.load('embeddings/project_embeddings.npy')
pro_ids = pd.read_csv('embeddings/project_ids.csv')

print(pro_emb.shape)
print(len(pro_ids))

(35830, 384)
35830
